In [72]:
from openai import OpenAI 
import json
import networkx as nx
import ipycytoscape
import pandas as pd
import os
import math
import re
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore", category= DeprecationWarning)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)


In [73]:
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
llm_model_name = "deepseek-ai/DeepSeek-V3-0324"

In [74]:
load_dotenv()

LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
client = OpenAI(
    # Sets the base URL for the API endpoint to the Nebius service.
    base_url=LITELLM_API_BASE,
    # Sets the API key for authentication. Replace with your actual key, preferably loaded from a secure source.
    api_key=NEBIUS_API_KEY
)

In [75]:
llm_temperature = 0.0
llm_max_tokens = 4096

In [76]:
unstructured_text = """
For the history of the British Isles before the formation of the United Kingdom, see History of the British Isles.
"UK History" redirects here. For the UKTV channel formerly known as UK History, see U&Yesterday.

A published version of the Treaty of Union agreement, which led to the creation of the Kingdom of Great Britain in 1707
This article is part of a series on the
History of the
United Kingdom

Timeline
Topics
Places
OutlineList of yearsHistoriography Category Portal
vte
The history of the United Kingdom begins in 1707 with the Treaty of Union and Acts of Union. The core of the United Kingdom as a unified state came into being with the political union of the kingdoms of England and Scotland,[1] into a new unitary state called Great Britain.[a]

The first decades were marked by Jacobite risings which ended with defeat for the Stuart cause at the Battle of Culloden in 1746. In 1763, victory in the Seven Years' War led to the growth of the First British Empire. With defeat by the US, France and Spain in the War of American Independence, Great Britain lost its 13 American colonies and rebuilt a Second British Empire based in Asia and Africa. Politically the central event was the French Revolution and its Napoleonic aftermath from 1793 to 1815, which British elites saw as a profound threat, and worked energetically to form multiple coalitions that finally defeated Napoleon in 1815. The Acts of Union 1800 added the Kingdom of Ireland to create the United Kingdom of Great Britain and Ireland.

The Tories, who came to power in 1783, remained in power until 1830. Forces of reform opened decades of political reform that broadened the ballot, and opened the economy to free trade. The outstanding political leaders of the 19th century included Palmerston, Disraeli, Gladstone, and Salisbury. Culturally, the Victorian era was a time of prosperity and dominant middle-class virtues when Britain dominated the world economy and maintained a generally peaceful century from 1815 to 1914. The First World War, with Britain in alliance with France, Russia and the US, was a furious but ultimately successful total war with Germany. The resulting League of Nations was a favourite project in Interwar Britain. In 1922, 26 counties of Ireland seceded to become the Irish Free State; a day later, Northern Ireland seceded from the Free State and returned to the United Kingdom. In 1927, the United Kingdom changed its formal title to the United Kingdom of Great Britain and Northern Ireland,[2] usually shortened to Britain, United Kingdom or UK. While the Empire remained strong, as did the London financial markets, the British industrial base began to slip behind Germany and the US. Sentiments for peace were so strong that the nation supported appeasement of Hitler's Germany in the 1930s, until the Nazi invasion of Poland in 1939 started the Second World War. In the Second World War, the Soviet Union and the US joined the UK as the main Allied powers.

After the war, Britain was no longer a military or economic superpower, as seen in the Suez Crisis of 1956. Britain granted independence to almost all its possessions. The new states typically joined the Commonwealth of Nations. The postwar years saw great hardships, alleviated somewhat by large-scale financial aid from the US. Prosperity returned in the 1950s. Meanwhile, from 1945 to 1950, the Labour Party built a welfare state, nationalised many industries, and created the National Health Service. The UK took a strong stand against Communist expansion after 1945, playing a major role in the Cold War and the formation of NATO as an anti-Soviet military alliance with West Germany, France, the US, Italy, Canada and smaller countries. The UK has been a leading member of the United Nations since its founding, as well as other international organisations. In the 1990s, neoliberalism led to the privatisation of nationalised industries and significant deregulation of business affairs. London's status as a world financial hub grew. Since the 1990s, large-scale devolution movements in Northern Ireland, Scotland and Wales have decentralised political decision-making. Britain has moved back and forth on its economic relationships with Western Europe. It joined the European Economic Community in 1973, thereby weakening economic ties with its Commonwealth. However, the Brexit referendum in 2016 committed the UK to leave the European Union, which it did in 2020.
"""

In [77]:
char_count = len(unstructured_text)
word_count = len(unstructured_text.split())

print("Word count", word_count)
print("Char count", char_count)

print("Unstructured Text \n", unstructured_text)

Word count 724
Char count 4452
Unstructured Text 
 
For the history of the British Isles before the formation of the United Kingdom, see History of the British Isles.
"UK History" redirects here. For the UKTV channel formerly known as UK History, see U&Yesterday.

A published version of the Treaty of Union agreement, which led to the creation of the Kingdom of Great Britain in 1707
This article is part of a series on the
History of the
United Kingdom

Timeline
Topics
Places
OutlineList of yearsHistoriography Category Portal
vte
The history of the United Kingdom begins in 1707 with the Treaty of Union and Acts of Union. The core of the United Kingdom as a unified state came into being with the political union of the kingdoms of England and Scotland,[1] into a new unitary state called Great Britain.[a]

The first decades were marked by Jacobite risings which ended with defeat for the Stuart cause at the Battle of Culloden in 1746. In 1763, victory in the Seven Years' War led to the growt

In [78]:
#Chunking Configuration

chunk_size = 150
chunk_overlap = 30

if chunk_overlap > chunk_size and chunk_size > 0:
    raise SystemExit(f"Erorr: Overlap {chunk_overlap} must be smaller than chunk size {chunk_size}")
else:
    print("Chunnking condig is valid")

Chunnking condig is valid


In [79]:
words = unstructured_text.split()
total_words = len(words)

In [80]:
chunks = []
start_index = 0
chunk_number = 1

while start_index < total_words:
    end_index = min(start_index + chunk_size, total_words)
    chunk_text = " ".join(words[start_index: end_index])
    chunks.append({"text":chunk_text, "chunk_number": chunk_number})

    next_start_index = start_index + chunk_size - chunk_overlap

    if next_start_index <= start_index:
        if end_index == total_words:
            break
        next_start_index = start_index + 1

    start_index = next_start_index
    chunk_number += 1

    if chunk_number > total_words:
        break
    


In [81]:
if chunks:
    chunks_df = pd.DataFrame(chunks)
    chunks_df["word_count"] = chunks_df["text"].apply(lambda x: len(x.split()))
    display(chunks_df[["chunk_number", "word_count", "text"]])

else:
    print("No chunks were created")
    

,chunk_number,word_count,text
0,1,150,"For the history of the British Isles before the formation of the United Kingdom, see History of the British Isles. ""UK History"" redirects here. Fo..."
1,2,150,"and Scotland,[1] into a new unitary state called Great Britain.[a] The first decades were marked by Jacobite risings which ended with defeat for t..."
2,3,150,"Acts of Union 1800 added the Kingdom of Ireland to create the United Kingdom of Great Britain and Ireland. The Tories, who came to power in 1783, ..."
3,4,150,"League of Nations was a favourite project in Interwar Britain. In 1922, 26 counties of Ireland seceded to become the Irish Free State; a day later..."
4,5,150,"the Second World War, the Soviet Union and the US joined the UK as the main Allied powers. After the war, Britain was no longer a military or econ..."
5,6,124,"as an anti-Soviet military alliance with West Germany, France, the US, Italy, Canada and smaller countries. The UK has been a leading member of th..."
6,7,4,it did in 2020.


## SYSTEM AND USER PROMPTS 

In [82]:
extraction_system_prompt = """
You are an AI expert specialized in knowledge graph extraction. Your task is to identify and extract factual Subject-Predicate-Object (SPO) triples from the given text. 
Focus on accuracy and adhere strictly to JSON output format request in the user prompt. Extract core entities and the most direct relationship
"""

extraction_user_prompt_template = """

Please extract Subject-Predicate-Object (SPO) triples from the text below.

**VERY IMPORTANT RULES:**
1.  **Output Format:** Respond ONLY with a single, valid JSON array. Each element MUST be an object with keys "subject", "predicate", "object".
2.  **JSON Only:** Do NOT include any text before or after the JSON array (e.g., no 'Here is the JSON:' or explanations). Do NOT use markdown ```json ... ``` tags.
3.  **Concise Predicates:** Keep the 'predicate' value concise (1-3 words, ideally 1-2). Use verbs or short verb phrases (e.g., 'discovered', 'was born in', 'won').
4.  **Lowercase:** ALL values for 'subject', 'predicate', and 'object' MUST be lowercase.
5.  **Pronoun Resolution:** Replace pronouns (she, he, it, her, etc.) with the specific lowercase entity name they refer to based on the text context (e.g., 'United Kingdom').
6.  **Specificity:** Capture specific details (e.g., 'Island in the Europe' instead of just 'Island' if specified).
7.  **Completeness:** Extract all distinct factual relationships mentioned.

**Text to Process:**
```text
{text_chunk}
```

**Required JSON Output Format Example:**
[
  {{ "subject": "Tories", "predicate": "rules", "object": "United Kingdom" }},
  {{ "subject": "Labour Party", "predicate": "built", "object": "welfare state" }}
]

**Your JSON Output (MUST start with '[' and end with ']'):**
"""

In [83]:
all_extracted_triples = []
failed_chunks = []

In [84]:
chunk_index = 0

if chunk_index < len(chunks):
    chunk_info = chunks[chunk_index]
    chunk_text = chunk_info["text"]
    chunk_num = chunk_info["chunk_number"]

    user_prompt = extraction_user_prompt_template.format(text_chunk = chunk_text)
    llm_output = None
    error_message = None

    try:
        response = client.chat.completions.create(
            model = llm_model_name,
            messages = [
                {"role": "system", "content": extraction_system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        llm_output = response.choices[0].message.content.strip()
        print(llm_output)

    except Exception as e:
        print("exception",e)
        error_message = str(e)
        failed_chunks.append({"chunk_number": chunk_num, "error": f"API/Processing error: {error_message}", "response": " "})

    parsed_json = None
    parsing_error = None
    if llm_output is not None:
        try:
            parsed_data = json.loads(llm_output)

            if isinstance(parsed_data, dict):
                list_values = [v for v in parsed_data.values() if isinstance(v,list)]
                if len(list_values) == 1:
                    parsed_json = list_values[0]
                else:
                    raise ValueError("JSON object received, but does not contain single list of triples") 
            elif isinstance(parsed_data, list):
                parsed_json = parsed_data
            else:
                raise ValueError("Parsed JSON is not a list or expected dictionary wrapper")

        except json.JSONDecodeError as json_error:
            parsing_error = f"JSONDecodeError: {json_error}. Trying regex fallback..."
            match = re.search(r'^\s*(\[.*?\])\s*$', llm_output, re.DOTALL)
            if match:
                json_string_extracted = match.group(1)

                try:
                    parsed_json = json.loads(json_string_extracted)
                    parsing_error = None
                except json.JSONDecodeError as nested_err:
                    print("regex content is not valid error")
                    print(f"      ERROR: Regex content is not valid JSON: {nested_err}")

            else:
                parsing_error = "JSONDecodeError and Regex fallback failed."
        
        except ValueError as val_err:
            parsing_error = f"Value Error: {val_err}"


    if parsed_json is not None:
        valid_triples_in_chunk = []
        invalid_entries = []
        if isinstance(parsed_json, list):
            for item in parsed_json:
                if isinstance(item, dict) and all(k in item for k in ["subject", "predicate", "object"]):
                    if all(isinstance(item[k], str) for k in ['subject', 'predicate', 'object']):
                        item['chunk'] = chunk_num
                        valid_triples_in_chunk.append(item)
                    else:
                        invalid_entries.append({"item": item, "reason": "Non-string value"})
                else:
                    invalid_entries.append({"item": item, "reason": "Incorrect structure/ keys"})
        
        else:
            invalid_entries({"item" : parsed_json, "reason": "Not a list"})

            if not any(fc["chunk_number"] == chunk_num for fc in failed_chunks):
                failed_chunks.append({"chunk_number": chunk_num, "error": "Parsed data not a list", "response": llm_output})

        if invalid_entries:
            print(f"Skipped {len(invalid_entries)} invalid entries")

        if valid_triples_in_chunk:
            display(pd.DataFrame(valid_triples_in_chunk))
            all_extracted_triples.extend(valid_triples_in_chunk)
        
        else:
            print("sdfsd")

else:
    print(f"chunk index {chunk_index} is out of bounds: (Total CHUkns): {len(chunks)}")


[
  { "subject": "united kingdom", "predicate": "formed by", "object": "treaty of union" },
  { "subject": "united kingdom", "predicate": "formed by", "object": "acts of union" },
  { "subject": "united kingdom", "predicate": "started in", "object": "1707" },
  { "subject": "great britain", "predicate": "created by", "object": "political union" },
  { "subject": "great britain", "predicate": "comprises", "object": "england" },
  { "subject": "great britain", "predicate": "comprises", "object": "scotland" },
  { "subject": "stuart cause", "predicate": "defeated at", "object": "battle" }
]


,subject,predicate,object,chunk
0,united kingdom,formed by,treaty of union,1
1,united kingdom,formed by,acts of union,1
2,united kingdom,started in,1707,1
3,great britain,created by,political union,1
4,great britain,comprises,england,1
5,great britain,comprises,scotland,1
6,stuart cause,defeated at,battle,1


## Normalize and De-deplicate Triples

In [85]:
normalized_triples = []
seen_triples = set()
original_count = len(all_extracted_triples)
empty_removed_count = 0
duplicates_removed_count = 0

In [88]:

example_limit = 5
processed_count = 0

for i,triple in enumerate(all_extracted_triples):
    show_example = (i<example_limit)
    if show_example:
        print(f" Example {i+1}")

    subject_raw = triple.get("subject")
    predicate_raw = triple.get("predicate")
    object_raw = triple.get("object")
    chunk_num = triple.get("chunk", "unknown")

    triple_valid = False

    normalized_sub, normalized_pred, normalized_obj = None, None, None

    if isinstance(subject_raw, str) and isinstance(predicate_raw, str) and isinstance(object_raw, str):
        normalized_sub = subject_raw.strip().lower()
        normalized_pred = re.sub(r'\s+', '', predicate_raw.strip().lower()).strip()
        normalized_obj = object_raw.strip().lower()

        if normalized_sub and normalized_obj and normalized_pred:
            triple_identifier = (normalized_sub, normalized_obj, normalized_pred)

            if triple_identifier not in seen_triples:
                normalized_triples.append({
                    'subject': normalized_sub,
                    'predicate': normalized_pred,
                    'object': normalized_obj,
                    'source_chunk': chunk_num

                })
                seen_triples.add(triple_identifier)
                triple_valid = True
            
            else:
                duplicates_removed_count += 1
    
    else:
        empty_removed_count += 1
    
    processed_count += 1
        

 Example 1
 Example 2
 Example 3
 Example 4
 Example 5


In [89]:
if normalized_triples:
    normalized_df = pd.DataFrame(normalized_triples)
    display(normalized_df)
else:
    print("No valid")


,subject,predicate,object,source_chunk
0,united kingdom,formedby,treaty of union,1
1,united kingdom,formedby,acts of union,1
2,united kingdom,startedin,1707,1
3,great britain,createdby,political union,1
4,great britain,comprises,england,1
5,great britain,comprises,scotland,1
6,stuart cause,defeatedat,battle,1


In [90]:

knowledge_graph = nx.DiGraph()
added_edges_count = 0
update_intervals = 5

if not normalized_triples:
    print("No normalized triples added to graph")

else:
    for i,triple in enumerate(normalized_triples):
        subject_node = triple['subject']
        object_node = triple['object']
        predicate_label = triple['predicate']


        knowledge_graph.add_edge(subject_node, object_node, label = predicate_label)
        added_edges_count += 1

        if (i+1) % update_intervals == 0 or (i+1) == len(normalized_triples):
            print("")
            try:
                print(nx.info(knowledge_graph))
            except:
                print(f"Type: {type(knowledge_graph).__name__}")



Type: DiGraph

Type: DiGraph


In [92]:
num_nodes = knowledge_graph.number_of_nodes()
num_edges = knowledge_graph.number_of_edges()

if num_edges != added_edges_count and isinstance(knowledge_graph, nx.DiGraph):
    print(f"Note: Added{added_edges_count} edges, graph has {num_edges}")


if num_nodes > 0:
    try:
        density = nx.density(knowledge_graph)
        if nx.is_weakly_connected(knowledge_graph):
            print("weakly connected graph")

        else:
            num_components = nx.number_weakly_connected_components(knowledge_graph)
    except Exception as e:
        print(f"Could not calculated some graph metrices: {e}")

else:
    print("graph is empty")


if num_nodes > 0:
    nodes_sample = list(knowledge_graph.nodes())[:10]
    display(pd.DataFrame(nodes_sample, columns=["Node Sample"]))


if num_edges > 0:
    edges_sample = []
    for u, v, data in list(knowledge_graph.edges(data=True))[:10]:  # move [:10] outside list()
        edges_sample.append({"Source": u, "Target": v, "Label": data.get("label", "N/A")})
else:
    print("Graph has no edges")
    


,Node Sample
0,united kingdom
1,treaty of union
2,acts of union
3,1707
4,great britain
5,political union
6,england
7,scotland
8,stuart cause
9,battle


In [93]:
can_visualize = False
if 'knowledge_graph' not in locals() or not isinstance(knowledge_graph, nx.Graph):
    print("Error")
elif knowledge_graph.number_of_nodes() == 0:
    print("NetworkX graph is empty")
else:
    can_visualize = True

In [94]:
cytoscape_nodes = []
cytoscape_edges = []

if can_visualize:
    print("Converting nodes...")
    # Calculate degrees for node sizing
    node_degrees = dict(knowledge_graph.degree())
    max_degree = max(node_degrees.values()) if node_degrees else 1
    
    for node_id in knowledge_graph.nodes():
        degree = node_degrees.get(node_id, 0)
        # Simple scaling for node size (adjust logic as needed)
        node_size = 15 + (degree / max_degree) * 50 if max_degree > 0 else 15
        
        cytoscape_nodes.append({
            'data': {
                'id': str(node_id), # ID must be string
                'label': str(node_id).replace(' ', '\n'), # Display label (wrap spaces)
                'degree': degree,
                'size': node_size,
                'tooltip_text': f"Entity: {str(node_id)}\nDegree: {degree}" # Tooltip on hover
            }
        })
    print(f"Converted {len(cytoscape_nodes)} nodes.")
    
    print("Converting edges...")
    edge_count = 0
    for u, v, data in knowledge_graph.edges(data=True):
        edge_id = f"edge_{edge_count}" # Unique edge ID
        predicate_label = data.get('label', '')
        cytoscape_edges.append({
            'data': {
                'id': edge_id,
                'source': str(u),
                'target': str(v),
                'label': predicate_label, # Label on edge
                'tooltip_text': f"Relationship: {predicate_label}" # Tooltip on hover
            }
        })
        edge_count += 1
    print(f"Converted {len(cytoscape_edges)} edges.")
    
    # Combine into the final structure
    cytoscape_graph_data = {'nodes': cytoscape_nodes, 'edges': cytoscape_edges}
    
    # Visualize the converted structure (first few nodes/edges)
    print("\n--- Sample Cytoscape Node Data (First 2) ---")
    print(json.dumps(cytoscape_graph_data['nodes'][:2], indent=2))
    print("\n--- Sample Cytoscape Edge Data (First 2) ---")
    print(json.dumps(cytoscape_graph_data['edges'][:2], indent=2))
    print("-" * 25)
else:
     print("Skipping data conversion as graph is not valid for visualization.")
     cytoscape_graph_data = {'nodes': [], 'edges': []}

Converting nodes...
Converted 10 nodes.
Converting edges...
Converted 7 edges.

--- Sample Cytoscape Node Data (First 2) ---
[
  {
    "data": {
      "id": "united kingdom",
      "label": "united\nkingdom",
      "degree": 3,
      "size": 65.0,
      "tooltip_text": "Entity: united kingdom\nDegree: 3"
    }
  },
  {
    "data": {
      "id": "treaty of union",
      "label": "treaty\nof\nunion",
      "degree": 1,
      "size": 31.666666666666664,
      "tooltip_text": "Entity: treaty of union\nDegree: 1"
    }
  }
]

--- Sample Cytoscape Edge Data (First 2) ---
[
  {
    "data": {
      "id": "edge_0",
      "source": "united kingdom",
      "target": "treaty of union",
      "label": "formedby",
      "tooltip_text": "Relationship: formedby"
    }
  },
  {
    "data": {
      "id": "edge_1",
      "source": "united kingdom",
      "target": "acts of union",
      "label": "formedby",
      "tooltip_text": "Relationship: formedby"
    }
  }
]
-------------------------


In [95]:
if can_visualize:
    print("Creating ipycytoscape widget...")
    cyto_widget = ipycytoscape.CytoscapeWidget()
    print("Widget created.")
    
    print("Loading graph data into widget...")
    cyto_widget.graph.add_graph_from_json(cytoscape_graph_data, directed=True)
    print("Data loaded.")
else:
    print("Skipping widget creation.")
    cyto_widget = None

Creating ipycytoscape widget...
Widget created.
Loading graph data into widget...
Data loaded.
